**Test NVIDIA GPU Availability**

In [ ]:
!nvidia-smi

# 1. Install Requirements

In [ ]:
!pip install ultralytics

# 2. Input Image Dataset and Prepare Traing Data

Always connect Google Drive Frist

In [ ]:
# Connect to google drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Shortcut path
datasets_path = '/path/to/authorized-data-or-backup'
custom_data_path = '/content/custom_data'
backup_model_path = '/path/to/authorized-data-or-backup'

In [ ]:
# Unzip images to a custom data folder
!unzip -q "$datasets_path" -d "$custom_data_path"

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="$custom_data_path" --train_pct=0.9

# 3. Configure Training

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

In [ ]:
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt

# --- CẤU HÌNH ĐƯỜNG DẪN TẠI ĐÂY ---
YAML_PATH = '/content/data.yaml'
TRAIN_LABELS = '/content/data/train/labels'
VAL_LABELS = '/content/data/validation/labels'
# ----------------------------------

def summarize_weld_dataset():
    # 1. Đọc tên class từ file yaml
    with open(YAML_PATH, 'r') as f:
        data_cfg = yaml.safe_load(f)

    raw_names = data_cfg['names']
    if isinstance(raw_names, list):
        class_names = {i: name for i, name in enumerate(raw_names)}
    else:
        class_names = raw_names

    def count_in_dir(label_dir):
        stats = {name: 0 for name in class_names.values()}
        img_count = 0
        if not os.path.exists(label_dir):
            print(f"❌ Không tìm thấy: {label_dir}")
            return stats, 0

        for file in os.listdir(label_dir):
            if file.endswith('.txt'):
                img_count += 1
                with open(os.path.join(label_dir, file), 'r') as f:
                    for line in f:
                        parts = line.split()
                        if parts:
                            c_id = int(parts[0])
                            c_name = class_names.get(c_id, f"Unknown_{c_id}")
                            if c_name not in stats: stats[c_name] = 0
                            stats[c_name] += 1
        return stats, img_count

    # 2. Thống kê
    train_stats, train_n = count_in_dir(TRAIN_LABELS)
    val_stats, val_n = count_in_dir(VAL_LABELS)

    # 3. Hiển thị bảng
    df = pd.DataFrame([train_stats, val_stats], index=['Train', 'Validation']).T
    df['Total'] = df['Train'] + df['Validation']

    print(f"\n✅ Đã tìm thấy {train_n} ảnh Train và {val_n} ảnh Validation.")
    print("-" * 50)
    print(df)
    print("-" * 50)

    # 4. Vẽ biểu đồ
    df[['Train', 'Validation']].plot(kind='bar', stacked=True, figsize=(10, 5), color=['#3498db', '#e74c3c'])
    plt.title('Thống kê số lượng mẫu (Instances) của từng lớp lỗi')
    plt.ylabel('Số lượng')
    plt.xticks(rotation=30)
    plt.grid(axis='y', alpha=0.3)
    plt.show()

summarize_weld_dataset()

In [ ]:
import os
import yaml
import shutil
import math
from collections import Counter
import pandas as pd

# --- CẤU HÌNH ĐƯỜNG DẪN ---
YAML_PATH = '/content/data.yaml'
TRAIN_IMG_DIR = '/content/data/train/images'
TRAIN_LABEL_DIR = '/content/data/train/labels'
MAX_DUP_LIMIT = 50
UNKNOWN_CLASS_NAME = "Unknown" # Tên lớp bạn muốn gộp vào

def get_stats(label_dir, class_names):
    count = Counter()
    total_files = len([f for f in os.listdir(label_dir) if f.endswith('.txt')])

    for label_file in os.listdir(label_dir):
        if label_file.endswith('.txt'):
            with open(os.path.join(label_dir, label_file), 'r') as f:
                classes = [int(line.split()[0]) for line in f if line.strip()]
                count.update(classes)

    readable_stats = {class_names.get(i, f"ID_{i}"): c for i, c in count.items()}
    return readable_stats, total_files

def auto_balance_with_unknown_merge():
    # 1. Đọc cấu hình YAML
    with open(YAML_PATH, 'r') as f:
        data_cfg = yaml.safe_load(f)
    raw_names = data_cfg['names']
    class_names = {i: name for i, name in enumerate(raw_names)} if isinstance(raw_names, list) else raw_names

    # Tìm ID của lớp Unknown
    unknown_id = None
    for i, name in class_names.items():
        if name == UNKNOWN_CLASS_NAME:
            unknown_id = i
            break

    if unknown_id is None:
        print(f"❌ Lỗi: Không tìm thấy lớp '{UNKNOWN_CLASS_NAME}' trong file yaml!")
        return

    # 2. Dọn dẹp dữ liệu cũ & Gộp các mẫu lạ vào Unknown
    print(f"🧹 Đang dọn dẹp file cũ và gộp các nhãn lạ vào ID {unknown_id} ({UNKNOWN_CLASS_NAME})...")
    valid_ids = set(class_names.keys())

    for f in os.listdir(TRAIN_LABEL_DIR):
        if f.endswith('.txt'):
            label_path = os.path.join(TRAIN_LABEL_DIR, f)

            # Xóa file nhân bản cũ
            if '_auto' in f:
                os.remove(label_path)
                continue

            # Sửa nội dung file nhãn gốc nếu có ID lạ
            new_lines = []
            modified = False
            with open(label_path, 'r') as file:
                lines = file.readlines()
                for line in lines:
                    parts = line.split()
                    if parts:
                        cls_id = int(parts[0])
                        if cls_id not in valid_ids:
                            parts[0] = str(unknown_id)
                            modified = True
                        new_lines.append(' '.join(parts) + '\n')

            if modified:
                with open(label_path, 'w') as file:
                    file.writelines(new_lines)

    for f in os.listdir(TRAIN_IMG_DIR):
        if '_auto' in f: os.remove(os.path.join(TRAIN_IMG_DIR, f))

    # 3. Thống kê TRƯỚC khi balance (Sau khi đã gộp Unknown)
    stats_before, img_before = get_stats(TRAIN_LABEL_DIR, class_names)

    # 4. Tính toán Logic nhân bản
    label_files = [f for f in os.listdir(TRAIN_LABEL_DIR) if f.endswith('.txt')]
    class_counts_list = Counter()
    image_to_classes = {}

    for label_file in label_files:
        path = os.path.join(TRAIN_LABEL_DIR, label_file)
        with open(path, 'r') as f:
            classes = [int(line.split()[0]) for line in f if line.strip()]
            if classes:
                class_counts_list.update(classes)
                image_to_classes[label_file] = list(set(classes))

    max_val = max(class_counts_list.values())
    dup_factors = {cls: min(MAX_DUP_LIMIT, math.ceil(max_val / count))
                   for cls, count in class_counts_list.items()}

    # 5. Thực hiện nhân bản
    print("🚀 Đang thực hiện cân bằng dữ liệu tự động...")
    for label_file, classes in image_to_classes.items():
        max_f = max([dup_factors[c] for c in classes])
        if max_f > 1:
            base_name = label_file.replace('.txt', '')
            img_ext = None
            for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
                if os.path.exists(os.path.join(TRAIN_IMG_DIR, base_name + ext)):
                    img_ext = ext
                    break
            if img_ext:
                for i in range(1, max_f):
                    new_name = f"{base_name}_auto{i}"
                    shutil.copy(os.path.join(TRAIN_LABEL_DIR, label_file), os.path.join(TRAIN_LABEL_DIR, new_name + '.txt'))
                    shutil.copy(os.path.join(TRAIN_IMG_DIR, base_name + img_ext), os.path.join(TRAIN_IMG_DIR, new_name + img_ext))

    # 6. Thống kê SAU khi balance
    stats_after, img_after = get_stats(TRAIN_LABEL_DIR, class_names)

    # 7. IN BÁO CÁO
    print("\n" + "="*65)
    print(f"{'BÁO CÁO CÂN BẰNG DATASET (ĐÃ GỘP UNKNOWN)':^65}")
    print("="*65)

    report_data = []
    for cls in class_names.values():
        b = stats_before.get(cls, 0)
        a = stats_after.get(cls, 0)
        factor = f"x{math.ceil(a/b)}" if b > 0 else "0"
        report_data.append([cls, b, a, factor])

    df_report = pd.DataFrame(report_data, columns=['Lớp lỗi', 'Trước (mẫu)', 'Sau (mẫu)', 'Tỉ lệ tăng'])
    print(df_report.to_string(index=False))

    print("-" * 65)
    print(f"Tổng số lượng ảnh thực tế: Trước: {img_before} -> Sau: {img_after}")
    print("="*65)
    print("✅ Xong! Các ID lạ đã được đưa về Unknown và Dataset đã cân bằng.")

# Chạy script
auto_balance_with_unknown_merge()

# 4. Run Training

First train

In [ ]:
!yolo segment train data=/content/data.yaml \
model=yolo26m-seg.pt \
epochs=150 \
imgsz=1024 \
batch=8 \
cls=2 \
device=0 \
lr0=0.001 \
lrf=0.01 \
momentum=0.937 \
weight_decay=0.0005 \
warmup_epochs=3 \
warmup_bias_lr=0.1 \
warmup_momentum=0.8 \
dropout=0.2 \
label_smoothing=0.15 \
hsv_h=0.015 \
hsv_s=0.7 \
hsv_v=0.4 \
degrees=15 \
translate=0.1 \
scale=0.5 \
flipud=0.5 \
fliplr=0.5 \
mosaic=1.0 \
mixup=0.1 \
patience=50 \
save_period=10 \
save=True \
plots=True

Option 1: Resume Train (If it has been interrupted unexpectedly)

In [ ]:
!unzip -q "$backup_model_path/my_model.zip"  -d /content/my_backup_model

In [ ]:
!yolo segment train resume model=/content/my_model/last.pt

Option 2: Resume Train (Train continue with new datasets)

In [ ]:
!yolo segment train \
data=/content/data.yaml \
model=/content/runs/segment/train/weights/best.pt \
epochs=100 \
imgsz=1024

# 5. Test Model

In [ ]:
!yolo segment predict \
model=runs/segment/train/weights/best_v5.1.pt \
source=data/validation/images \
save=True \
show_conf=False \
line_width=2

In [ ]:
!yolo segment predict \
model="/path/to/local/project-artifact" \
source="/path/to/local/project-artifact" \
save=True \
project="/path/to/local/project-artifact" \
name="test-3" \
exist_ok=True \
show_conf=False \
show_labels=False \
line_width=3

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/segment/predict/*.jpg')[:40]:
  display(Image(filename=image_path, height=400))
  print('\n')

# 6. Backup YOLO Model

In [ ]:
!mkdir -p /content/my_model
!cp /content/runs/segment/train/weights/best.pt /content/my_model/best.pt
!cp /content/runs/segment/train/weights/last.pt /content/my_model/last.pt

%cd /content
!zip -r /content/my_model.zip my_model

In [ ]:
!cp /content/my_model.zip "$backup_model_path"